In [1]:
# ====================================================================
# 👑 NOTEBOOK : POISSON SUPREMACY (THE PERFECT ADAPTATION)
# Stratégie : Aggregation (Trials/Events) + Tweedie 1.5 + Weighted Training
# ====================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# 1. CHARGEMENT & FEATURE ENGINEERING
print("⏳ Chargement...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    # Features Sniper Elite
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

# On prépare les données brutes
train_eng = feature_engineering(train_data)
test_eng = feature_engineering(test_data)

# 2. L'AGRÉGATION (La clé de la perfection)
# --------------------------------------------------------------------
print("🔨 Agrégation des profils similaires...")

# On définit les colonnes qui définissent un "Profil Unique"
features = ['country', 'age', 'new_user', 'source', 'total_pages_visited', 
            'interaction_age_pages', 'pages_per_age']

# Petite astuce : on arrondit les floats pour être sûr que le groupby fonctionne bien
train_eng['pages_per_age'] = train_eng['pages_per_age'].round(6)
test_eng['pages_per_age'] = test_eng['pages_per_age'].round(6)

# GroupBy sur le Train pour créer le dataset "Pur"
# On compte les essais (count) et les succès (sum)
agg_train = train_eng.groupby(features).agg(
    n_trials=('converted', 'count'),
    conversion_rate=('converted', 'mean') # Cible = Taux de conversion (0 à 1)
).reset_index()

print(f"📉 Réduction du dataset : {len(train_eng)} lignes -> {len(agg_train)} profils uniques.")

# 3. PRÉPARATION
# --------------------------------------------------------------------
X_agg = agg_train[features]
y_agg = agg_train['conversion_rate'] # On prédit une probabilité
w_agg = agg_train['n_trials']        # POIDS CRUCIAL : On pondère par le nombre d'essais

X_test = test_eng[features]

# Encodage
categorical_cols = ['country', 'source']
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X_agg[col], X_test[col]])
    le.fit(combined)
    X_agg[col] = le.transform(X_agg[col])
    X_test[col] = le.transform(X_test[col])

# 4. ENTRAÎNEMENT MASSIF (Bagging sur Agrégé)
# --------------------------------------------------------------------
# Puisque c'est très rapide (13k lignes), on peut faire 50 seeds !
N_FOLDS = 10
N_SEEDS = 5 # Tu peux monter à 10 ou 20 si tu as le temps
BEST_RHO = 1.5 # Le paramètre magique Tweedie

print(f"🚀 Lancement de la Supremacy ({N_FOLDS} Folds x {N_SEEDS} Seeds)...")

# Attention : Pour la validation, on doit utiliser les données BRUTES (Raw)
# pour que le score soit comparable au Leaderboard.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# On doit mapper les indices bruts vers les indices agrégés... c'est complique.
# Stratégie plus simple : On entraîne sur tout l'agrégé avec variation de seed, 
# et on valide OOF en reconstruisant.
# Mais pour faire simple et robuste ici : 
# On va faire un Bagging simple sur l'ensemble du dataset agrégé.

dtrain_full = xgb.DMatrix(X_agg, label=y_agg, weight=w_agg) # Note le weight !
dtest_full = xgb.DMatrix(X_test)

test_preds_accumulated = np.zeros(len(X_test))

for i in range(N_SEEDS):
    seed = 42 + i*100
    params = {
        'objective': 'reg:tweedie',
        'tweedie_variance_power': BEST_RHO,
        'eval_metric': 'logloss',
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': seed
    }
    
    # Entraînement sur les données agrégées pondérées
    model = xgb.train(params, dtrain_full, num_boost_round=600, verbose_eval=False)
    
    # Prédiction
    test_preds_accumulated += model.predict(dtest_full)
    print(f"  > Seed {seed} terminée.")

# Moyenne
test_preds_avg = test_preds_accumulated / N_SEEDS

# 5. GÉNÉRATION
# --------------------------------------------------------------------
# On reprend le seuil optimal calculé précédemment (0.372) car la distribution est la même
# Ou on peut le recalculer si on avait fait une CV, mais 0.372 est très robuste pour Tweedie/Poisson.
FINAL_THRESHOLD = 0.372191 

sub = pd.DataFrame({'converted': (test_preds_avg >= FINAL_THRESHOLD).astype(int)})
filename = 'submission_POISSON_SUPREMACY.csv'
sub.to_csv(filename, index=False)

print(f"\n✅ Fichier '{filename}' généré sur la base de {len(agg_train)} profils purs.")
print(f"📊 Conversions trouvées : {sub['converted'].sum()}")

⏳ Chargement...
🔨 Agrégation des profils similaires...
📉 Réduction du dataset : 284580 lignes -> 13942 profils uniques.
🚀 Lancement de la Supremacy (10 Folds x 5 Seeds)...
  > Seed 42 terminée.
  > Seed 142 terminée.
  > Seed 242 terminée.
  > Seed 342 terminée.
  > Seed 442 terminée.

✅ Fichier 'submission_POISSON_SUPREMACY.csv' généré sur la base de 13942 profils purs.
📊 Conversions trouvées : 935


In [2]:
from sklearn.metrics import f1_score

print("\n🔍 Calcul du score local (Cross-Validation)...")

# On définit le seuil optimal trouvé précédemment
THRESHOLD = 0.372191

# Initialisation des prédictions Out-Of-Fold
oof_preds = np.zeros(len(X_agg))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# On utilise le taux de conversion binarisé pour stratifier
strat_target = (y_agg > 0).astype(int)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_agg, strat_target)):
    # Split
    X_tr, X_val = X_agg.iloc[train_idx], X_agg.iloc[val_idx]
    y_tr, y_val = y_agg.iloc[train_idx], y_agg.iloc[val_idx]
    w_tr, w_val = w_agg.iloc[train_idx], w_agg.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dval = xgb.DMatrix(X_val)
    
    # Même paramètres
    params = {
        'objective': 'reg:tweedie',
        'tweedie_variance_power': 1.5,
        'learning_rate': 0.05,
        'max_depth': 6,
        'seed': 42
    }
    
    m = xgb.train(params, dtrain, num_boost_round=600)
    oof_preds[val_idx] = m.predict(dval)

# CALCUL DU SCORE F1
# Important : On doit "re-multiplier" par les poids pour simuler le dataset complet
y_true_raw = []
y_pred_raw = []

for i in range(len(agg_train)):
    # On recrée 'n_trials' lignes pour chaque profil
    n = int(w_agg.iloc[i])
    actual_conv = int(y_agg.iloc[i] * n) # Nombre de convertis réels
    
    # Liste de 1 et 0 réels
    y_true_raw.extend([1] * actual_conv + [0] * (n - actual_conv))
    
    # Prédiction binarisée répétée n fois
    pred_bin = 1 if oof_preds[i] >= THRESHOLD else 0
    y_pred_raw.extend([pred_bin] * n)

final_f1 = f1_score(y_true_raw, y_pred_raw)
print(f"🚀 SCORE F1 ESTIMÉ : {final_f1:.5f}")


🔍 Calcul du score local (Cross-Validation)...
🚀 SCORE F1 ESTIMÉ : 0.76666
